# Agente de carga académica: demostración paso a paso

Sistema Multiagente (Historial, Carga, Horarios y Manager) coordinado con **Contract Net**, auditado con **Shapley** y con una **ontología OWL** como base de conocimiento.

> **Datos:** el plan de estudios es el **real** de Ingeniería en Sistemas Computacionales de la Ibero Puebla (79 materias exportadas de Intrauia, con su seriación). La oferta de grupos, los profesores y los alumnos son **sintéticos**.

> Kernel: usa el entorno `.venv` de esta carpeta (Python 3.12).
> El LLM es opcional. Si quieres usarlo, ejecuta antes `./iniciar_ollama_cpu.sh` en una terminal: así Ollama corre **solo en CPU**.

In [1]:
from carga.kb import BaseConocimiento
from carga.llm import ClienteLLM, diagnostico
from carga import sistema

print("\n".join(diagnostico()))
USAR_LLM = False          # cámbialo a True si levantaste ./iniciar_ollama_cpu.sh
kb = BaseConocimiento()   # carga la ontología poblada UNA sola vez
llm = ClienteLLM(habilitado=USAR_LLM)

✔ Ollama responde en http://127.0.0.1:11435 — modelos: llama3.1:8b, llama3.2:latest, gemma3:1b
✔ Ollama responde en http://127.0.0.1:11434 — modelos: NINGUNO
  RAM disponible: 7.2 GB de 14.9 GB
  Energía: cargador conectado ✔ — batería 95%, 8.46 V (mín. de diseño 7.64 V)
  Modelo configurado: llama3.2 | hilos: 4 | contexto: 4096


## 1. Base de conocimiento: el alumno según la ontología (consultado por SPARQL)

In [2]:
est = kb.estudiante("A001")
print(est.nombre, "| semestre", est.semestre, "| promedio", est.promedio)
print("Aprobadas:", ", ".join(est.aprobadas))
print("Reprobadas:", est.reprobadas)
print("Preferencias guardadas:", est.preferencias)

Ana Torres Medina | semestre 5 | promedio 8.78
Aprobadas: LIIM2101, LIIE2102, LIIS1201, LIIS2101, LICB1219, LICB2101, LIIB2101, LICB1224, LICB1202, LICB2103, LICB2107, LHLF1201, LICB1231, LICB2106, LIIB2102, LICB2105, LIIM2102, LICB2109, LICB2108, LYRU2101, LICB0423, LICB2110, LICB2111, LIIS2111, LIIB2103, LIIS2102, LYRU2105
Reprobadas: {'LIIS2105': 5.4}
Preferencias guardadas: {'turno': ('vespertino', 3), 'carga': ('normal', 1, 40, 7), 'modalidad': ('presencial', 2)}


## 2. AgenteHorarios interpreta la petición en lenguaje natural

In [3]:
peticion = "Me interesa inteligencia artificial, prefiero la tarde y no quiero clases los viernes"
s = sistema.nueva_sesion("A001", peticion, usar_llm=USAR_LLM, eco=True, kb=kb, llm=llm)
from carga.preferencias import describir
print(describir(s.prefs), "\n→", s.prefs["interpretado_por"])

carga normal (~40 créditos, máx. 7 materias); turno vespertino; modalidad presencial; sin clases: Viernes; le interesa: LIIS2109 
→ reglas (sin LLM)


## 3. Contract Net: cfp → propose → accept/reject

Compiten **3 AgentesCarga** (CargaAvance, CargaEquilibrio y CargaPreferencias). Cada uno oferta sus 2 mejores cargas según su estrategia y el Manager las califica con la función oficial.

In [4]:
res = sistema.correr_contract_net(s)
for p in res.propuestas:
    print(p.id, p.autor, p.puntuacion, p.materias, p.criterios)
print("\nGanador del contrato:", res.ganadora.autor)

  [      cfp       ] AgenteManager → AgenteHistorial: ¿Qué materias puede cursar A001?
  [    propose     ] AgenteHistorial → AgenteManager: válidas: LIIS2105, LIIS2112, LIIB2104, LICB2102, LIIE2104, LIIS2113, LIIS2114, LIIN2103, LIIS2106, LIIS2108, LIIS2117, LICB1229, LIIE2103, LIII0428, LIIN2105, LIIN2106, LIIS2109, LIIS2120, LRSS2102, LYRU2118, LYRU2104, LYRU2110, LYRU2111, LYRU2113, LIIN2110, LIIS2122, LIIS2123, LIIS2124
  [      cfp       ] AgenteManager → CargaAvance: Arma una carga de 24-48 créditos, meta 40
  [      cfp       ] AgenteManager → CargaEquilibrio: Arma una carga de 24-48 créditos, meta 40
  [      cfp       ] AgenteManager → CargaPreferencias: Arma una carga de 24-48 créditos, meta 40
  [    propose     ] CargaAvance → AgenteManager: 2 ofertas; la mejor (autoevaluación 0.67): LIIS2109, LIIS2105, LIIS2112, LICB2102, LIIE2104, LIIS2114
  [    propose     ] CargaEquilibrio → AgenteManager: 2 ofertas; la mejor (autoevaluación 0.858): LIIS2109, LIIS2105, LIIS2112, LIIB2

## 4. Transparencia: valores de Shapley (exactos, 8 coaliciones)

In [5]:
from carga.reporte import _coal
phi = sistema.correr_shapley(s)
for S, v in sorted(s.v.items(), key=lambda x: len(x[0])):
    print(f"v({_coal(S)}) = {v}")
print("\nφ:", phi, "\nsuma:", round(sum(phi.values()), 2), "= v(N)")

v({}) = 0.0
v({A}) = 71.88
v({B}) = 60.06
v({C}) = 59.07
v({A,B}) = 81.52
v({A,C}) = 71.88
v({B,C}) = 69.21
v({A,B,C}) = 93.01

φ: {'AgenteHistorial': 37.6, 'AgenteCarga': 30.36, 'AgenteHorarios': 25.04} 
suma: 93.0 = v(N)


### 4.1 Shapley entre los 3 contratistas

Segundo juego: Historial y Horarios siempre están; juegan solo los AgentesCarga. v(S) se mide contra la carga ingenua, para que v(∅) = 0. Si dos contratistas ofertan lo mismo, Shapley les paga lo mismo (**simetría**).

In [6]:
from carga.reporte import _coal_c, licitacion
for c, n, mejor, auto, gano in licitacion(res):
    print(f"{c:18} ofertas={n}  mejor oficial={mejor}  autoevaluación={auto}  {'GANA' if gano else ''}")
print()
for S, v in sorted(s.v_c.items(), key=lambda x: len(x[0])):
    print(f"v({_coal_c(S)}) = {v}")
print("\nφ contratistas:", s.phi_c, "\nsuma:", round(sum(s.phi_c.values()), 2), "= v(N)")

CargaAvance        ofertas=2  mejor oficial=93.01  autoevaluación=0.67  GANA
CargaEquilibrio    ofertas=2  mejor oficial=91.23  autoevaluación=0.858  
CargaPreferencias  ofertas=2  mejor oficial=91.81  autoevaluación=0.931  

v({}) = 0.0
v({Avance}) = 21.13
v({Equilibrio}) = 19.35
v({Preferencias}) = 19.93
v({Avance, Equilibrio}) = 21.13
v({Avance, Preferencias}) = 21.13
v({Equilibrio, Preferencias}) = 19.93
v({Avance, Equilibrio, Preferencias}) = 21.13

φ contratistas: {'CargaAvance': 7.94, 'CargaEquilibrio': 6.45, 'CargaPreferencias': 6.74} 
suma: 21.13 = v(N)


## 5. Razonador HermiT sobre las clases definidas

In [7]:
for clase, miembros in kb.inferencias_hermit().items():
    print(f"{clase:20} {miembros}")

MateriaLlave         ['LIIS2105', 'LIIS2112', 'LYRU2101']
MateriaSeriada       ['LICB2103', 'LICB2105', 'LICB2108', 'LIIB2102', 'LIIB2103', 'LIIB2104', 'LIIM2102', 'LIIN1203', 'LIIN2104', 'LIIN2107', 'LIIS2107', 'LIIS2110', 'LIIS2113', 'LIIS2115', 'LIIS2116', 'LIIS2117', 'LIIS2118', 'LIIS2119', 'LIIS2121', 'LIIS2125', 'LRSS2102', 'LYRU1213', 'LYRU2102', 'LYRU2103', 'LYRU2104', 'LYRU2105', 'LYRU2106', 'LYRU2107', 'LYRU2108', 'LYRU2109', 'LYRU2110', 'LYRU2111', 'LYRU2112', 'LYRU2113', 'LYRU2118']
GrupoVespertino      ['LHLF1201-A', 'LICB1202-B', 'LICB1229-A', 'LICB1231-A', 'LICB2102-A', 'LICB2103-B', 'LICB2106-A', 'LICB2107-A', 'LICB2108-A', 'LICB2110-A', 'LICB2111-A', 'LIIB2102-A', 'LIIB2103-B', 'LIIB2104-B', 'LIIE2103-B', 'LIIE2104-B', 'LIII0428-B', 'LIIN1203-A', 'LIIN2103-B', 'LIIN2104-A', 'LIIN2105-A', 'LIIN2106-B', 'LIIN2107-A', 'LIIN2110-A', 'LIIS2102-B', 'LIIS2105-A', 'LIIS2106-A', 'LIIS2107-B', 'LIIS2108-A', 'LIIS2109-B', 'LIIS2110-A', 'LIIS2111-A', 'LIIS2112-B', 'LIIS2113-B', 'L

## 6. Recomendación final, reporte y resultados escritos de vuelta en la ontología

In [8]:
rutas = sistema.cerrar(s)
from IPython.display import Markdown, display
display(Markdown(rutas["reporte"].read_text(encoding="utf-8")))


══════════════════════════════════════════════════════════════════════════════
 Ana Torres Medina (A001) · semestre 5 · Primavera 2027
 Preferencias: carga normal (~40 créditos, máx. 7 materias); turno vespertino; modalidad presencial; sin clases: Viernes; le interesa: LIIS2109
   ↳ interpretadas por: reglas (sin LLM)
══════════════════════════════════════════════════════════════════════════════
 CARGA RECOMENDADA (40 créditos · puntuación 93.01/100) · la ofertó CargaAvance
  • LIIS2109 Fundamentos de Inteligencia Arti LIIS2109-B presencial Mar 17-19, Jue 17-19
  • LIIS2105 Programación Avanzada y Algoritm LIIS2105-A presencial Lun 17-19, Mie 17-19
  • LIIS2112 Diseño y Gestión de Bases de Dat LIIS2112-A presencial Lun 07-09, Mie 07-09
  • LICB2102 Estadística y Probabilidad       LICB2102-A presencial Lun 19-21, Mie 19-21
  • LIIS2113 Diseño de Software II            LIIS2113-B hibrida    Mar 19-21, Jue 19-21
  • LIIS2114 Fundamentos de Sistemas Operativ LIIS2114-A en_linea   Mar 07-

# Recomendación de carga académica — Ana Torres Medina (A001)

**Periodo:** Primavera 2027 · **Semestre:** 5 · **Promedio:** 8.78 · **Créditos acumulados:** 144

## Preferencias interpretadas

- Petición: _Me interesa inteligencia artificial, prefiero la tarde y no quiero clases los viernes_
- Interpretado por: reglas (sin LLM)
- Resultado: carga normal (~40 créditos, máx. 7 materias); turno vespertino; modalidad presencial; sin clases: Viernes; le interesa: LIIS2109

| Campo | Valor | Fuente |
|---|---|---|
| tipo_carga | normal | ontología |
| turno | vespertino | petición (reglas) |
| modalidad | presencial | ontología |
| dias_libres | ['Viernes'] | petición (reglas) |
| materias_interes | ['LIIS2109'] | petición (reglas) |

## Recomendación final

Ana, para Primavera 2027 te recomendamos inscribir 6 materias (40 créditos): Fundamentos de Inteligencia Artificial, Programación Avanzada y Algoritmos, Diseño y Gestión de Bases de Datos I, Estadística y Probabilidad, Diseño de Software II, Fundamentos de Sistemas Operativos. Se prioriza recursar Programación Avanzada y Algoritmos porque es prerrequisito de materias posteriores. La combinación cumple el rango institucional de 24–48 créditos y obtuvo 93.01/100 en la evaluación del coordinador. No hay choques de horario y el ajuste a tus preferencias de turno, modalidad e intereses es de 89.8/100.

_Redactado por: plantilla (sin LLM)_

**Propuesta ganadora P2** (ofertada por **CargaAvance**) — 40 créditos · puntuación **93.01/100**

| Clave | Materia | Créd. | Grupo | Modalidad | Horario | Profesor |
|---|---|---|---|---|---|---|
| LIIS2109 | Fundamentos de Inteligencia Artificial | 6 | LIIS2109-B | presencial | Mar 17-19, Jue 17-19 | Mtra. Gabriela Ortiz |
| LIIS2105 | Programación Avanzada y Algoritmos (recursa) | 6 | LIIS2105-A | presencial | Lun 17-19, Mie 17-19 | Mtro. Daniel Fuentes |
| LIIS2112 | Diseño y Gestión de Bases de Datos I | 8 | LIIS2112-A | presencial | Lun 07-09, Mie 07-09 | Mtra. Laura Peña |
| LICB2102 | Estadística y Probabilidad | 6 | LICB2102-A | presencial | Lun 19-21, Mie 19-21 | Mtra. Laura Peña |
| LIIS2113 | Diseño de Software II | 8 | LIIS2113-B | hibrida | Mar 19-21, Jue 19-21 | Mtro. Ricardo Solís |
| LIIS2114 | Fundamentos de Sistemas Operativos | 6 | LIIS2114-A | en linea | Mar 07-09, Jue 07-09 | Dra. Elena Márquez |

### Horario semanal

| Hora | Lun | Mar | Mie | Jue | Vie |
|---|---|---|---|---|---|
| 07:00 | LIIS2112 | LIIS2114 | LIIS2112 | LIIS2114 |  |
| 08:00 | LIIS2112 | LIIS2114 | LIIS2112 | LIIS2114 |  |
| 09:00 |  |  |  |  |  |
| 10:00 |  |  |  |  |  |
| 11:00 |  |  |  |  |  |
| 12:00 |  |  |  |  |  |
| 13:00 |  |  |  |  |  |
| 14:00 |  |  |  |  |  |
| 15:00 |  |  |  |  |  |
| 16:00 |  |  |  |  |  |
| 17:00 | LIIS2105 | LIIS2109 | LIIS2105 | LIIS2109 |  |
| 18:00 | LIIS2105 | LIIS2109 | LIIS2105 | LIIS2109 |  |
| 19:00 | LICB2102 | LIIS2113 | LICB2102 | LIIS2113 |  |
| 20:00 | LICB2102 | LIIS2113 | LICB2102 | LIIS2113 |  |

## Agente de Historial: materias descartadas

- **LIII2106** Tópicos de Ingeniería Industrial: no se oferta en Primavera 2027
- **LIII2107** Tópicos en Gestión de Tecnología: no se oferta en Primavera 2027
- **LIIL2102** Tópicos de Ingeniería en Logística: no se oferta en Primavera 2027
- **LIIN1203** Implementación y Evaluación de Proyectos: falta prerrequisito: LIIS2117
- **LIIN2104** Proyecto de Innovación Frugal: falta prerrequisito: LIIB2104
- **LIIN2107** Práctica Profesional II: falta prerrequisito: LIIN2103
- **LIIS2103** Tópicos en Gestión de la Información: no se oferta en Primavera 2027
- **LIIS2104** Control Remoto e Internet: no se oferta en Primavera 2027
- **LIIS2107** Sistemas Embebidos II: falta prerrequisito: LIIS2106
- **LIIS2110** Paradigmas de Programación Avanzados: falta prerrequisito: LIIS2105
- **LIIS2115** Diseño y Gestión de Bases de Datos II: falta prerrequisito: LIIS2112
- **LIIS2116** Programación de Dispositivos Móviles: falta prerrequisito: LIIS2105
- **LIIS2118** Cómputo en la Nube: falta prerrequisito: LIIS2112
- **LIIS2119** Programación de Aplicaciones y Servicios Web: falta prerrequisito: LIIS2105
- **LIIS2121** Programación de Ambientes Virtuales y Juegos: falta prerrequisito: LIIS2105
- **LIIS2125** Redes Digitales Avanzadas: falta prerrequisito: LIIS2108
- **LYRU1213** Fe en América Latina: no se oferta en Primavera 2027
- **LYRU2102** Experiencia Estética: no se oferta en Primavera 2027
- **LYRU2103** Narrativas Corporales: no se oferta en Primavera 2027
- **LYRU2106** Historia de Dios: no se oferta en Primavera 2027
- **LYRU2107** Espiritualidad Hoy: no se oferta en Primavera 2027
- **LYRU2108** Religión en la Sociedad Actual: no se oferta en Primavera 2027
- **LYRU2109** Medio Universitario: no se oferta en Primavera 2027
- **LYRU2112** Comunidades Afectivas: no se oferta en Primavera 2027

## Contract Net: licitación entre los AgentesCarga

El Manager anunció la misma tarea a 3 contratistas. Cada uno ofertó sus 2 mejores cargas según su estrategia (autoevaluación) y el Manager las calificó con la función oficial.

| Contratista | Pesos (carga/prioridad/interés/horario/facilidad) | Ofertas | Mejor puntuación oficial | Su autoevaluación | Resultado |
|---|---|---|---|---|---|
| CargaAvance | 0.3/0.6/0.1/0.0/0.0 | 2 | 93.01 | 0.67 | **gana** ✅ |
| CargaEquilibrio | 0.5/0.1/0.1/0.0/0.3 | 2 | 91.23 | 0.858 | rechazado |
| CargaPreferencias | 0.3/0.1/0.3/0.3/0.0 | 2 | 91.81 | 0.931 | rechazado |

Las autoevaluaciones **no se comparan entre sí**: cada contratista se califica con su propia escala. Por eso el Manager adjudica con la función oficial y no con lo que cada uno dice de sí mismo.

## Contract Net: propuestas evaluadas

| Propuesta | Ofertó | Materias | Créd. | Avance (0.30) | Horario (0.25) | Carga (0.25) | Preferencias (0.20) | **Total** | Notas |
|---|---|---|---|---|---|---|---|---|---|
| P2 ✅ | Avance | LIIS2109, LIIS2105, LIIS2112, LICB2102, LIIS2113, LIIS2114 | 40 | 83.5 | 100 | 100.0 | 89.8 | **93.01** |  |
| P1 | Avance | LIIS2109, LIIS2105, LIIS2112, LICB2102, LIIE2104, LIIS2114 | 40 | 83.5 | 100 | 100.0 | 87.5 | **92.55** |  |
| P6 | Preferencias | LIIS2109, LIIS2105, LIIS2112, LIIB2104, LICB2102, LIIS2117 | 38 | 81.4 | 100 | 95.0 | 93.2 | **91.81** |  |
| P3 | Equilibrio | LIIS2109, LIIS2105, LIIS2112, LIIB2104, LICB2102, LIIS2114, LIIN2103 | 40 | 83.1 | 100 | 100.0 | 81.5 | **91.23** |  |
| P4 | Equilibrio | LIIS2109, LIIS2105, LIIB2104, LICB2102, LIIE2104, LIIS2114, LIIN2103 | 40 | 82.4 | 100 | 100.0 | 81.5 | **91.02** |  |
| P5 | Preferencias | LIIS2109, LIIS2105, LIIS2112, LIIB2104, LICB2102, LIIE2104 | 38 | 81.7 | 100 | 95.0 | 81.8 | **89.62** |  |

## Transparencia: valores de Shapley

v(S) = puntuación de la mejor propuesta que logra la coalición S (A = Historial, B = Carga, C = Horarios). Cuando un agente no participa, su tarea se hace de forma ingenua.

| Coalición S | v(S) |
|---|---|
| {} | 0.0 |
| {A} | 71.88 |
| {B} | 60.06 |
| {C} | 59.07 |
| {A,B} | 81.52 |
| {A,C} | 71.88 |
| {B,C} | 69.21 |
| {A,B,C} | 93.01 |

| Agente | φ (Shapley) | % de la contribución |
|---|---|---|
| AgenteHistorial (A) | 37.6 | 40.4% |
| AgenteCarga (B) | 30.36 | 32.6% |
| AgenteHorarios (C) | 25.04 | 26.9% |
| **Suma** | **93.0** = v(N) | 100% |

B es el **equipo** de los 3 contratistas: cuando B no participa, la carga se arma de forma ingenua.

### Shapley entre los contratistas

Segundo juego: Historial y Horarios siempre presentes; juegan solo los 3 AgentesCarga. v(S) = puntuación con los contratistas de S **menos** la puntuación sin ninguno (carga ingenua), para que v(∅) = 0.

| Coalición S | v(S) |
|---|---|
| {} | 0.0 |
| {Avance} | 21.13 |
| {Equilibrio} | 19.35 |
| {Preferencias} | 19.93 |
| {Avance, Equilibrio} | 21.13 |
| {Avance, Preferencias} | 21.13 |
| {Equilibrio, Preferencias} | 19.93 |
| {Avance, Equilibrio, Preferencias} | 21.13 |

| Contratista | φ |
|---|---|
| CargaAvance | 7.94 |
| CargaEquilibrio | 6.45 |
| CargaPreferencias | 6.74 |
| **Suma** | **21.13** = v(N) |

## Mensajes del Contract Net (FIPA-ACL)

```
[      cfp       ] AgenteManager → AgenteHistorial: ¿Qué materias puede cursar A001?
[    propose     ] AgenteHistorial → AgenteManager: válidas: LIIS2105, LIIS2112, LIIB2104, LICB2102, LIIE2104, LIIS2113, LIIS2114, LIIN2103, LIIS2106, LIIS2108, LIIS2117, LICB1229, LIIE2103, LIII0428, LIIN2105, LIIN2106, LIIS2109, LIIS2120, LRSS2102, LYRU2118, LYRU2104, LYRU2110, LYRU2111, LYRU2113, LIIN2110, LIIS2122, LIIS2123, LIIS2124
[      cfp       ] AgenteManager → CargaAvance: Arma una carga de 24-48 créditos, meta 40
[      cfp       ] AgenteManager → CargaEquilibrio: Arma una carga de 24-48 créditos, meta 40
[      cfp       ] AgenteManager → CargaPreferencias: Arma una carga de 24-48 créditos, meta 40
[    propose     ] CargaAvance → AgenteManager: 2 ofertas; la mejor (autoevaluación 0.67): LIIS2109, LIIS2105, LIIS2112, LICB2102, LIIE2104, LIIS2114
[    propose     ] CargaEquilibrio → AgenteManager: 2 ofertas; la mejor (autoevaluación 0.858): LIIS2109, LIIS2105, LIIS2112, LIIB2104, LICB2102, LIIS2114, LIIN2103
[    propose     ] CargaPreferencias → AgenteManager: 2 ofertas; la mejor (autoevaluación 0.931): LIIS2109, LIIS2105, LIIS2112, LIIB2104, LICB2102, LIIE2104
[      cfp       ] AgenteManager → AgenteHorarios: Asignar grupos a 6 combinaciones
[    propose     ] AgenteHorarios → AgenteManager: 6 propuestas con horario
[accept-proposal ] AgenteManager → CargaAvance: P2 (93.01): LIIS2109, LIIS2105, LIIS2112, LICB2102, LIIS2113, LIIS2114
[reject-proposal ] AgenteManager → CargaAvance: P1 (92.55)
[reject-proposal ] AgenteManager → CargaPreferencias: P6 (91.81)
[reject-proposal ] AgenteManager → CargaEquilibrio: P3 (91.23)
[reject-proposal ] AgenteManager → CargaEquilibrio: P4 (91.02)
[reject-proposal ] AgenteManager → CargaPreferencias: P5 (89.62)
[     inform     ] CargaAvance → AgenteManager: carga adjudicada: 40 créditos, 6 materias
```

## Trazas de ejecución

| Hora | Agente | Tarea | Salida | Tiempo (s) | Éxito | Score |
|---|---|---|---|---|---|---|
| 21:11:41 | AgenteHorarios | interpretar_preferencias | reglas (sin LLM) | 0.005 | ✔ |  |
| 21:11:50 | AgenteHistorial | verificar_historial_y_seriacion | 28 válidas, 24 descartadas | 0.0001 | ✔ |  |
| 21:11:50 | CargaAvance | ofertar_carga | 2 ofertas (mejor autoevaluación 0.67) | 0.0319 | ✔ |  |
| 21:11:50 | CargaEquilibrio | ofertar_carga | 2 ofertas (mejor autoevaluación 0.858) | 0.0288 | ✔ |  |
| 21:11:50 | CargaPreferencias | ofertar_carga | 2 ofertas (mejor autoevaluación 0.931) | 0.0285 | ✔ |  |
| 21:11:50 | AgenteHorarios | detectar_conflictos_y_preferencias | 6 propuestas con grupos asignados | 0.0056 | ✔ |  |
| 21:11:50 | AgenteManager | evaluar_propuestas_contract_net | gana P2 con 93.01 | 0.0004 | ✔ | 93.01 |
| 21:11:54 | AgenteManager | calcular_valores_shapley | AgenteHistorial=37.6, AgenteCarga=30.36, AgenteHorarios=25.04 | 0.3932 | ✔ |  |
| 21:11:55 | AgenteManager | shapley_entre_contratistas | CargaAvance=7.94, CargaEquilibrio=6.45, CargaPreferencias=6.74 | 0.3774 | ✔ |  |
| 21:12:11 | AgenteManager | generar_recomendacion_final | plantilla (sin LLM) | 0.0001 | ✔ |  |

_LLM: no disponible · llamadas: 0 · tiempo LLM: 0.0 s · desactivado con --sin-llm_
